# 6자유도 유도탄 동역학 (6-DOF Missile Dynamics)

**참고문헌 (References)**
- Zipfel, P. H., *Modeling and Simulation of Aerospace Vehicle Dynamics*, AIAA, 3rd ed. (Ch. 4–6)
- Stevens, B. L. & Lewis, F. L., *Aircraft Control and Simulation*, Wiley, 3rd ed.
- NOAA/NASA/USAF, *U.S. Standard Atmosphere 1976*

---

## 3DOF 질점 모델의 한계 → 자세 동역학이 필요한 이유

3자유도(3-DOF) 질점 모델은 유도탄을 질량이 집중된 점으로 취급하여 병진 운동만 기술합니다.
이 단순화에는 다음과 같은 근본적인 한계가 있습니다:

| 항목 | 3DOF 모델 | 6DOF 모델 |
|------|-----------|----------|
| 자세(Attitude) | 없음 — 속도벡터 방향 = 기체 방향 가정 | Quaternion으로 완전 추적 |
| 공력(Aerodynamics) | $C_L(\alpha)$, $C_D(\alpha)$ 만 사용 | 힘 + 모멘트 계수 (6개 축) |
| 조종면(Control surface) | 즉각적인 힘 명령 | 핀 편향 → 모멘트 → 자세변화 → 힘 |
| 자이로스코픽 결합 | 없음 | $\boldsymbol{\omega} \times (\mathbf{I}\boldsymbol{\omega})$ 포함 |
| 과도응답(Transient) | 없음 | 자세 정정 지연 반영 |

**실무적 함의:** 오토파일럿 설계, HILS 검증, 명중률 분석에서는 반드시 6DOF 모델이 필요합니다.
자세 과도응답 없이는 실제 비행궤적을 재현할 수 없기 때문입니다.

---

## 학습 목표

1. 좌표계 정의 및 방향여현행렬(DCM) 이해
2. **Quaternion 기반 6DOF 운동방정식(EOM)** 구현
3. US Standard Atmosphere 1976 모델 구현 및 검증
4. 선형 공력 모델 구현
5. RK4 수치적분으로 자유 비행 시뮬레이션
6. 3DOF vs 6DOF 궤적 비교

## 1. 좌표계 정의 (Coordinate Frame Definitions)

### 1.1 NED 관성좌표계 (North-East-Down Inertial Frame)

- $x_N$: 북쪽(North), $y_E$: 동쪽(East), $z_D$: 아래쪽(Down)
- **고도(altitude)** = $-z_D$ (양수 위쪽)
- 지구 곡률은 전술 유도탄 교전 거리($<$100 km)에서 무시 가능

### 1.2 동체좌표계 (Body Frame)

- $x_b$: 기체 전방(forward, 롤축)
- $y_b$: 기체 우측(right)
- $z_b$: 기체 하방(down)
- 각속도: $p$ (롤), $q$ (피치), $r$ (요)

### 1.3 바람좌표계 (Wind/Stability Frame)

- 받음각(Angle of Attack): $\alpha = \arctan(w/u)$
- 옆미끄럼각(Sideslip): $\beta = \arcsin(v/V)$
- 전속도: $V = \sqrt{u^2 + v^2 + w^2}$

### 1.4 오일러각과 짐벌락 문제

3-2-1 회전순서(요→피치→롤): $\psi \to \theta \to \phi$

방향여현행렬(DCM)은 다음과 같습니다:

$$
\mathbf{C}_{nb} = \mathbf{R}_1(\phi)\,\mathbf{R}_2(\theta)\,\mathbf{R}_3(\psi)
$$

$$
\mathbf{C}_{nb} = \begin{bmatrix}
c_\theta c_\psi & c_\theta s_\psi & -s_\theta \\
s_\phi s_\theta c_\psi - c_\phi s_\psi & s_\phi s_\theta s_\psi + c_\phi c_\psi & s_\phi c_\theta \\
c_\phi s_\theta c_\psi + s_\phi s_\psi & c_\phi s_\theta s_\psi - s_\phi c_\psi & c_\phi c_\theta
\end{bmatrix}
$$

여기서 $c_x = \cos x$, $s_x = \sin x$.

**짐벌락(Gimbal Lock) 문제:** $\theta = \pm 90°$일 때 $\phi$와 $\psi$가 독립적으로 결정되지 않아
수치적 특이점(singularity)이 발생합니다. 수직 발사나 급기동 유도탄에서 실제로 발생할 수 있어
쿼터니언(quaternion) 표현이 필수적입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

# ─────────────────────────────────────────────────────────────────────────────
# 1. 오일러각 → DCM  (3-2-1 sequence: yaw ψ → pitch θ → roll φ)
#    v_body = C_nb @ v_ned
# ─────────────────────────────────────────────────────────────────────────────
def euler_to_dcm(phi, theta, psi):
    """오일러각(φ,θ,ψ) → DCM  (NED → Body).
    Stevens & Lewis, eq. (1.3-20)
    """
    cphi, sphi = np.cos(phi),   np.sin(phi)
    cth,  sth  = np.cos(theta), np.sin(theta)
    cpsi, spsi = np.cos(psi),   np.sin(psi)

    R3 = np.array([[ cpsi, spsi, 0.0],
                   [-spsi, cpsi, 0.0],
                   [ 0.0,  0.0,  1.0]])
    R2 = np.array([[ cth, 0.0, -sth],
                   [ 0.0, 1.0,  0.0],
                   [ sth, 0.0,  cth]])
    R1 = np.array([[1.0,  0.0,   0.0 ],
                   [0.0,  cphi,  sphi],
                   [0.0, -sphi,  cphi]])
    return R1 @ R2 @ R3


def dcm_to_euler(C):
    """DCM → 오일러각(φ,θ,ψ) [rad].  θ = ±90° 근방 짐벌락 처리 포함."""
    sin_theta = np.clip(-C[0, 2], -1.0, 1.0)
    theta = np.arcsin(sin_theta)
    cos_theta = np.cos(theta)
    if abs(cos_theta) > 1e-10:
        phi = np.arctan2(C[1, 2], C[2, 2])
        psi = np.arctan2(C[0, 1], C[0, 0])
    else:                       # 짐벌락: φ=0 관례 적용
        phi = 0.0
        psi = np.arctan2(-C[1, 0], C[1, 1]) if sin_theta > 0 else np.arctan2(C[1, 0], -C[1, 1])
    return phi, theta, psi


# ─────────────────────────────────────────────────────────────────────────────
# 2. 쿼터니언 연산
# ─────────────────────────────────────────────────────────────────────────────
def quat_normalize(q):
    """단위 쿼터니언 정규화.  q = [q0, q1, q2, q3], q0 스칼라부."""
    n = np.linalg.norm(q)
    if n < 1e-12:
        raise ValueError(f"쿼터니언 노름이 너무 작음: {n}")
    return q / n


def quat_multiply(p, q):
    """Hamilton 곱  p ⊗ q  (scalar-first 규약).

    Stevens & Lewis, eq. (1.8-15):
        [p0 q0 - p·q,  p0 q_vec + q0 p_vec + p_vec × q_vec]
    """
    p0, p1, p2, p3 = p
    q0, q1, q2, q3 = q
    return np.array([
        p0*q0 - p1*q1 - p2*q2 - p3*q3,
        p0*q1 + p1*q0 + p2*q3 - p3*q2,
        p0*q2 - p1*q3 + p2*q0 + p3*q1,
        p0*q3 + p1*q2 - p2*q1 + p3*q0,
    ])


def quat_to_dcm(q):
    """단위 쿼터니언 → DCM  (NED → Body).

    Zipfel eq. (12.34), Stevens & Lewis eq. (1.8-16).
    """
    q = quat_normalize(q)
    q0, q1, q2, q3 = q
    return np.array([
        [q0**2+q1**2-q2**2-q3**2,  2*(q1*q2+q0*q3),          2*(q1*q3-q0*q2)          ],
        [2*(q1*q2-q0*q3),           q0**2-q1**2+q2**2-q3**2,  2*(q2*q3+q0*q1)          ],
        [2*(q1*q3+q0*q2),           2*(q2*q3-q0*q1),           q0**2-q1**2-q2**2+q3**2 ],
    ])


def euler_to_quat(phi, theta, psi):
    """오일러각 → 쿼터니언  (DCM 경유)."""
    C = euler_to_dcm(phi, theta, psi)
    trace = C[0,0] + C[1,1] + C[2,2]
    q_sq = np.maximum([
        (1.0 + trace) / 4.0,
        (1.0 + C[0,0] - C[1,1] - C[2,2]) / 4.0,
        (1.0 - C[0,0] + C[1,1] - C[2,2]) / 4.0,
        (1.0 - C[0,0] - C[1,1] + C[2,2]) / 4.0,
    ], 0.0)
    pivot = int(np.argmax(q_sq))
    if pivot == 0:
        q0 = np.sqrt(q_sq[0])
        q1 = (C[1,2]-C[2,1])/(4*q0);  q2 = (C[2,0]-C[0,2])/(4*q0);  q3 = (C[0,1]-C[1,0])/(4*q0)
    elif pivot == 1:
        q1 = np.sqrt(q_sq[1])
        q0 = (C[1,2]-C[2,1])/(4*q1);  q2 = (C[0,1]+C[1,0])/(4*q1);  q3 = (C[2,0]+C[0,2])/(4*q1)
    elif pivot == 2:
        q2 = np.sqrt(q_sq[2])
        q0 = (C[2,0]-C[0,2])/(4*q2);  q1 = (C[0,1]+C[1,0])/(4*q2);  q3 = (C[1,2]+C[2,1])/(4*q2)
    else:
        q3 = np.sqrt(q_sq[3])
        q0 = (C[0,1]-C[1,0])/(4*q3);  q1 = (C[2,0]+C[0,2])/(4*q3);  q2 = (C[1,2]+C[2,1])/(4*q3)
    q = np.array([q0, q1, q2, q3])
    if q[0] < 0.0:
        q = -q
    return quat_normalize(q)


def quat_to_euler(q):
    """쿼터니언 → 오일러각 (φ,θ,ψ) [rad]."""
    return dcm_to_euler(quat_to_dcm(q))


# ─────────────────────────────────────────────────────────────────────────────
# 검증 (Verification)
# ─────────────────────────────────────────────────────────────────────────────
phi_test, theta_test, psi_test = np.deg2rad(10.0), np.deg2rad(20.0), np.deg2rad(30.0)

C = euler_to_dcm(phi_test, theta_test, psi_test)
phi_r, theta_r, psi_r = dcm_to_euler(C)
q = euler_to_quat(phi_test, theta_test, psi_test)
C_from_q = quat_to_dcm(q)
phi_q, theta_q, psi_q = quat_to_euler(q)

print("=== DCM 직교성 검증 (C @ C^T ≈ I) ===")
print(np.round(C @ C.T, 10))
print(f"\n오일러각 왕복 변환 오차 [deg]:")
print(f"  φ: {np.rad2deg(phi_r-phi_test):.2e}  "
      f"θ: {np.rad2deg(theta_r-theta_test):.2e}  "
      f"ψ: {np.rad2deg(psi_r-psi_test):.2e}")
print(f"\n쿼터니언 왕복 변환 오차 [deg]:")
print(f"  φ: {np.rad2deg(phi_q-phi_test):.2e}  "
      f"θ: {np.rad2deg(theta_q-theta_test):.2e}  "
      f"ψ: {np.rad2deg(psi_q-psi_test):.2e}")
print(f"\n쿼터니언 q = {q}")
print(f"‖q‖ = {np.linalg.norm(q):.15f}")
print(f"\nDCM(오일러) vs DCM(쿼터니언) 최대 오차: {np.max(np.abs(C - C_from_q)):.2e}")

## 2. Quaternion 운동학 (Quaternion Kinematics)

### 쿼터니언 미분방정식

각속도 벡터 $\boldsymbol{\omega} = [p, q, r]^T$ (동체좌표계)에 대해 쿼터니언의 시간 미분은:

$$
\dot{\mathbf{q}} = \frac{1}{2}\,\mathbf{q} \otimes \boldsymbol{\Omega}
$$

여기서 $\boldsymbol{\Omega} = [0, p, q, r]^T$ (순수 쿼터니언).

4원소 전개 (scalar-first: $\mathbf{q} = [q_0, q_1, q_2, q_3]$):

$$
\begin{bmatrix} \dot{q}_0 \\ \dot{q}_1 \\ \dot{q}_2 \\ \dot{q}_3 \end{bmatrix}
= \frac{1}{2}
\begin{bmatrix}
 0  & -p & -q & -r \\
 p  &  0 &  r & -q \\
 q  & -r &  0 &  p \\
 r  &  q & -p &  0
\end{bmatrix}
\begin{bmatrix} q_0 \\ q_1 \\ q_2 \\ q_3 \end{bmatrix}
$$

**Euler angle 대비 singularity 없음** — 대부분의 현대 유도탄 시뮬레이션에서 사용.

4개 변수로 3자유도 회전을 기술하므로 단위제약 $‖\mathbf{q}‖=1$을 수치 적분마다 재정규화합니다.
이는 오일러각의 $\theta = \pm 90°$ 특이점을 완전히 회피합니다 (Stevens & Lewis §1.8).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 쿼터니언 운동학 구현 및 검증
# ─────────────────────────────────────────────────────────────────────────────

def quaternion_derivative(q, omega):
    """쿼터니언 시간 미분  q̇ = 0.5 * q ⊗ [0, p, q_r, r]

    Args:
        q     : [q0, q1, q2, q3]  단위 쿼터니언
        omega : [p, q, r]  동체 각속도 (rad/s)
    Returns:
        dq/dt : shape (4,)
    """
    q0, q1, q2, q3 = q
    p, q_r, r = omega

    # Omega matrix  (Zipfel eq. 12.37)
    dq0 = 0.5 * (-p*q1  - q_r*q2 - r*q3)
    dq1 = 0.5 * ( p*q0  + r*q2   - q_r*q3)
    dq2 = 0.5 * ( q_r*q0 - r*q1  + p*q3)
    dq3 = 0.5 * ( r*q0  + q_r*q1 - p*q2)
    return np.array([dq0, dq1, dq2, dq3])


# ── 테스트: 순수 요(yaw) 회전  ψ̇ = r = 0.1 rad/s ──────────────────────────
dt  = 0.001                          # 적분 스텝 [s]
t_end = 2 * np.pi / 0.1             # 1회전 완성 시간 [s]
t_arr = np.arange(0.0, t_end, dt)

omega_const = np.array([0.0, 0.0, 0.1])   # p=0, q=0, r=0.1 rad/s
q_cur = euler_to_quat(0.0, 0.0, 0.0)       # 초기자세: NED와 일치

psi_arr = []
norm_arr = []

for _ in t_arr:
    dq = quaternion_derivative(q_cur, omega_const)
    q_cur = q_cur + dt * dq
    q_cur = quat_normalize(q_cur)           # 단위제약 재정규화
    _, _, psi_cur = quat_to_euler(q_cur)
    psi_arr.append(np.rad2deg(psi_cur))
    norm_arr.append(np.linalg.norm(q_cur))

psi_analytical = np.rad2deg(omega_const[2] * t_arr)  # 해석해

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_arr, psi_analytical % 360, 'k--', lw=2, label='해석해 (analytical)')
axes[0].plot(t_arr, np.array(psi_arr) % 360, 'b-',  lw=1.5, label='쿼터니언 적분')
axes[0].set_xlabel('시간 [s]')
axes[0].set_ylabel('요각 ψ [deg]')
axes[0].set_title('순수 요 회전 추적 (r = 0.1 rad/s)')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(t_arr, norm_arr, 'r-', lw=1.5)
axes[1].axhline(1.0, color='k', ls='--', lw=1)
axes[1].set_xlabel('시간 [s]')
axes[1].set_ylabel('‖q‖')
axes[1].set_title('쿼터니언 단위제약 유지')
axes[1].set_ylim([0.9999, 1.0001])
axes[1].grid(True)

plt.tight_layout()
plt.show()

err_max = np.max(np.abs(np.array(psi_arr) % 360 - psi_analytical % 360))
print(f"최대 요각 오차: {err_max:.4f} deg")
print(f"쿼터니언 노름 최소/최대: {min(norm_arr):.12f} / {max(norm_arr):.12f}")

## 3. Newton-Euler 운동방정식 (Newton-Euler EOM)

### 13-상태 벡터

$$
\mathbf{x} = [\underbrace{x_N,\, y_E,\, z_D}_{\text{위치 (NED)}},\;
\underbrace{u,\, v,\, w}_{\text{속도 (동체)}},\;
\underbrace{q_0,\, q_1,\, q_2,\, q_3}_{\text{자세 (쿼터니언)}},\;
\underbrace{p,\, q,\, r}_{\text{각속도 (동체)}}]^T
$$

### 병진 운동방정식 (Translational EOM)

동체좌표계 Newton 제2법칙 (코리올리 항 포함):

$$
m\,\dot{\mathbf{V}}_b = \mathbf{F}_{aero} + \mathbf{F}_{thrust} + \mathbf{F}_{grav} - m\,\boldsymbol{\omega} \times \mathbf{V}_b
$$

성분별 전개:

$$
\begin{aligned}
m\dot{u} &= F_{x} + mg_x + m(rv - qw) \\
m\dot{v} &= F_{y} + mg_y + m(pw - ru) \\
m\dot{w} &= F_{z} + mg_z + m(qu - pv)
\end{aligned}
$$

중력의 동체 성분 (쿼터니언 직접 계산, Zipfel Ch.5):

$$
\begin{pmatrix} g_x \\ g_y \\ g_z \end{pmatrix}
= g\begin{pmatrix}
2(q_1 q_3 - q_0 q_2) \\
2(q_2 q_3 + q_0 q_1) \\
q_0^2 - q_1^2 - q_2^2 + q_3^2
\end{pmatrix}
$$

### 회전 운동방정식 (Rotational EOM, Euler's Equations)

$$
\mathbf{I}\dot{\boldsymbol{\omega}} = \mathbf{M}_{aero} - \boldsymbol{\omega} \times (\mathbf{I}\boldsymbol{\omega})
$$

축대칭 유도탄 ($I_{yy} = I_{zz}$)에 대해:

$$
\begin{aligned}
I_{xx}\dot{p} &= L + (I_{yy}-I_{zz})\,q\,r \\
I_{yy}\dot{q} &= M + (I_{zz}-I_{xx})\,r\,p \\
I_{zz}\dot{r} &= N + (I_{xx}-I_{yy})\,p\,q
\end{aligned}
$$

### 위치 운동학 (Position Kinematics)

$$
\dot{\mathbf{r}}_{NED} = \mathbf{C}_{bn}\,\mathbf{V}_b = \mathbf{C}_{nb}^T\,\mathbf{V}_b
$$

## 4. 공력 모델 (Aerodynamic Model)

### 힘 계수

무차원 동압: $\bar{q} = \frac{1}{2}\rho V^2$

| 계수 | 식 | 물리 의미 |
|------|----|-----------|
| $C_L = C_{L_\alpha}\,\alpha$ | 양력 | 받음각에 선형 비례 |
| $C_D = C_{D_0} + C_{D_{\alpha^2}}\,\alpha^2$ | 항력 | 영양력 항력 + 유도 항력 |
| $C_Y = C_{Y_\beta}\,\beta$ | 옆힘 | 축대칭: $C_{Y_\beta} = -C_{L_\alpha}$ |

동체축 힘 계수:

$$
C_X = -(C_D \cos\alpha - C_L \sin\alpha), \quad
C_Z = -(C_L \cos\alpha + C_D \sin\alpha)
$$

### 모멘트 계수 (정규화: $\bar{q}\,S_{ref}\,d_{ref}$)

정규화 각속도: $\hat{p} = pd_{ref}/(2V)$,  $\hat{q} = qd_{ref}/(2V)$,  $\hat{r} = rd_{ref}/(2V)$

$$
\begin{aligned}
C_m &= C_{m_\alpha}\,\alpha + C_{m_q}\,\hat{q} + C_{m_{\delta_e}}\,\delta_e \\
C_n &= C_{n_\beta}\,\beta  + C_{n_r}\,\hat{r} + C_{n_{\delta_r}}\,\delta_r \\
C_l &= C_{l_p}\,\hat{p}   + C_{l_{\delta_a}}\,\delta_a
\end{aligned}
$$

정적 안정성 조건: $C_{m_\alpha} < 0$ (피치), $C_{n_\beta} > 0$ (요).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 공력 모델 구현
# ─────────────────────────────────────────────────────────────────────────────

def wind_angles(u, v, w):
    """동체 속도 성분 → (α, β) [rad].

    α = atan2(w, u)        받음각 (positive nose-up)
    β = arcsin(v / V)     옆미끄럼각 (positive nose-right)
    """
    V = np.sqrt(u**2 + v**2 + w**2)
    alpha = np.arctan2(w, u)
    beta  = np.arcsin(np.clip(v / V, -1.0, 1.0)) if V > 1e-10 else 0.0
    return alpha, beta, V


class MissileAerodynamics:
    """선형 공력 계수 모델  (소~중간 받음각: |α|, |β| < ~20°).

    축대칭 전술 유도탄 가정:
        CY_beta = -CL_alpha,  Cn_beta = -Cm_alpha
    """

    def __init__(self):
        # 기준 기하
        self.S_ref = 0.01267   # m²  (π/4 × 0.127²)
        self.d_ref = 0.127     # m

        # 힘 계수
        self.CL_alpha  =  18.5   # /rad  양력 경사
        self.CD_0      =   0.35  # –     영양력 항력
        self.CD_alpha2 =   8.0   # /rad² 유도 항력
        self.CY_beta   = -18.5   # /rad  옆힘 (축대칭)

        # 모멘트 계수
        self.Cm_alpha   = -3.0   # /rad  피치 정적 안정성
        self.Cm_q       = -15.0  # /rad  피치 감쇠
        self.Cm_delta   = -1.2   # /rad  승강타 효과
        self.Cn_beta    =  3.0   # /rad  요 정적 안정성
        self.Cn_r       = -15.0  # /rad  요 감쇠
        self.Cn_delta_r = -1.2   # /rad  방향타 효과
        self.Cl_p       = -5.0   # /rad  롤 감쇠
        self.Cl_delta_a = -0.8   # /rad  에일러론 효과

    def get_forces_moments(self, alpha, beta, V, q_bar, p, q_r, r,
                           delta_e=0.0, delta_r=0.0, delta_a=0.0):
        """동체좌표계 공기력 및 공기 모멘트 계산.

        Returns:
            forces  : [Fx, Fy, Fz] (N)
            moments : [L, M, N]    (N·m)  — 롤, 피치, 요
        """
        qS  = q_bar * self.S_ref
        qSd = qS    * self.d_ref

        norm = (self.d_ref / (2.0 * V)) if V > 1e-6 else 0.0
        p_hat = p   * norm
        q_hat = q_r * norm
        r_hat = r   * norm

        # 힘 계수
        CL = self.CL_alpha * alpha
        CD = self.CD_0 + self.CD_alpha2 * alpha**2
        CX = -(CD * np.cos(alpha) - CL * np.sin(alpha))
        CZ = -(CL * np.cos(alpha) + CD * np.sin(alpha))
        CY =  self.CY_beta * beta

        forces = qS * np.array([CX, CY, CZ])

        # 모멘트 계수
        Cm = self.Cm_alpha * alpha + self.Cm_q * q_hat + self.Cm_delta   * delta_e
        Cn = self.Cn_beta  * beta  + self.Cn_r * r_hat + self.Cn_delta_r * delta_r
        Cl = self.Cl_p     * p_hat                     + self.Cl_delta_a * delta_a

        moments = qSd * np.array([Cl, Cm, Cn])
        return forces, moments


# ── 간단한 공력 특성 시각화 ────────────────────────────────────────────────
aero = MissileAerodynamics()
alphas = np.linspace(-20, 20, 200)
CL_arr = aero.CL_alpha * np.deg2rad(alphas)
CD_arr = aero.CD_0 + aero.CD_alpha2 * np.deg2rad(alphas)**2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(alphas, CL_arr, 'b-', lw=2, label='$C_L$')
axes[0].plot(alphas, CD_arr, 'r-', lw=2, label='$C_D$')
axes[0].set_xlabel('받음각 α [deg]')
axes[0].set_ylabel('공력 계수')
axes[0].set_title('양력 / 항력 계수')
axes[0].legend()
axes[0].grid(True)

# 양항비
LD = CL_arr / np.where(CD_arr > 0, CD_arr, np.nan)
axes[1].plot(alphas, LD, 'g-', lw=2)
axes[1].set_xlabel('받음각 α [deg]')
axes[1].set_ylabel('L/D')
axes[1].set_title('양항비 (L/D)')
axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f"최대 L/D = {np.nanmax(LD):.2f}  at  α = {alphas[np.nanargmax(LD)]:.1f} deg")

## 5. US Standard Atmosphere 1976

전술 유도탄 비행고도 범위 (0 – 32 km)에서의 표준 대기 모델.

| 층 | 고도 범위 | 온도 구배 |
|----|----------|-----------|
| 대류권 (Troposphere) | 0 – 11 km | −6.5 K/km |
| 대류권계면 (Tropopause) | 11 – 20 km | 0 (등온) |
| 하부 성층권 | 20 – 32 km | +1.0 K/km |

**온도 경사층 (gradient layer) 압력:**

$$P = P_b \left(\frac{T}{T_b}\right)^{-g_0/(\Lambda R)}$$

**등온층 (isothermal layer) 압력:**

$$P = P_b \exp\!\left(-\frac{g_0\,\Delta h}{R\,T_b}\right)$$

밀도 및 음속:
$$\rho = \frac{P}{RT}, \qquad a = \sqrt{\gamma R T}$$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# US Standard Atmosphere 1976 구현
# ─────────────────────────────────────────────────────────────────────────────

class StandardAtmosphere1976:
    """US Standard Atmosphere 1976  (0 – 32 km).

    NOAA/NASA/USAF 1976 문서 상수 사용.
    """
    R       = 287.0528   # J/(kg·K)
    g0      = 9.80665    # m/s²
    gamma   = 1.4
    T0      = 288.15     # K
    P0      = 101325.0   # Pa

    # (base_alt_m, base_T_K, lapse_K_per_m)
    LAYERS = [
        (0.0,      288.15, -0.0065),
        (11000.0,  216.65,  0.0),
        (20000.0,  216.65,  0.001),
        (32000.0,  228.65,  None),   # sentinel
    ]

    def __init__(self):
        # 각 층 기저 압력 사전 계산
        self._P_base = [self.P0]
        for i in range(len(self.LAYERS) - 1):
            h_b, T_b, lapse = self.LAYERS[i]
            h_top, _, _     = self.LAYERS[i+1]
            P_b = self._P_base[i]
            if lapse == 0.0:
                P_top = P_b * np.exp(-self.g0*(h_top-h_b) / (self.R*T_b))
            else:
                T_top = T_b + lapse*(h_top-h_b)
                P_top = P_b * (T_top/T_b) ** (-self.g0/(lapse*self.R))
            self._P_base.append(P_top)

    def _find_layer(self, h):
        for i in range(len(self.LAYERS)-2, -1, -1):
            if h >= self.LAYERS[i][0]:
                return i
        return 0

    def get_properties(self, altitude):
        """고도 → (T[K], P[Pa], ρ[kg/m³], a[m/s])."""
        h   = float(np.clip(altitude, 0.0, 32000.0))
        idx = self._find_layer(h)
        h_b, T_b, lapse = self.LAYERS[idx]
        P_b = self._P_base[idx]
        dh  = h - h_b
        if lapse == 0.0:
            T = T_b
            P = P_b * np.exp(-self.g0*dh / (self.R*T_b))
        else:
            T = T_b + lapse*dh
            P = P_b * (T/T_b) ** (-self.g0/(lapse*self.R))
        rho = P / (self.R * T)
        a   = np.sqrt(self.gamma * self.R * T)
        return T, P, rho, a


# ── 검증 플롯 ─────────────────────────────────────────────────────────────
atm    = StandardAtmosphere1976()
alts   = np.linspace(0, 32000, 500)
T_arr  = np.array([atm.get_properties(h)[0] for h in alts])
rho_arr= np.array([atm.get_properties(h)[2] for h in alts])
a_arr  = np.array([atm.get_properties(h)[3] for h in alts])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].plot(T_arr, alts/1000, 'r-', lw=2)
axes[0].axhline(11, color='gray', ls='--', lw=1, label='대류권계면 (11 km)')
axes[0].axhline(20, color='blue', ls='--', lw=1, label='성층권 (20 km)')
axes[0].set_xlabel('온도 T [K]')
axes[0].set_ylabel('고도 [km]')
axes[0].set_title('대기 온도 프로파일')
axes[0].legend(fontsize=8)
axes[0].grid(True)

axes[1].plot(rho_arr, alts/1000, 'b-', lw=2)
axes[1].axhline(11, color='gray', ls='--', lw=1)
axes[1].axhline(20, color='blue', ls='--', lw=1)
axes[1].set_xlabel('밀도 ρ [kg/m³]')
axes[1].set_ylabel('고도 [km]')
axes[1].set_title('대기 밀도 프로파일')
axes[1].grid(True)

axes[2].plot(a_arr, alts/1000, 'g-', lw=2)
axes[2].axhline(11, color='gray', ls='--', lw=1)
axes[2].axhline(20, color='blue', ls='--', lw=1)
axes[2].set_xlabel('음속 a [m/s]')
axes[2].set_ylabel('고도 [km]')
axes[2].set_title('음속 프로파일')
axes[2].grid(True)

plt.suptitle('US Standard Atmosphere 1976 (0 – 32 km)', fontsize=13)
plt.tight_layout()
plt.show()

# 표준값 대비 검증 (해수면)
T_sl, P_sl, rho_sl, a_sl = atm.get_properties(0.0)
print("=== 해수면 검증 (표준값 대비) ===")
print(f"온도:  {T_sl:.2f} K  (기준: 288.15 K)")
print(f"압력:  {P_sl:.1f} Pa (기준: 101325.0 Pa)")
print(f"밀도:  {rho_sl:.4f} kg/m³ (기준: 1.225 kg/m³)")
print(f"음속:  {a_sl:.2f} m/s (기준: 340.29 m/s)")

## 6. 6DOF 시뮬레이션 구현

### RK4 수치 적분

$$
\mathbf{x}_{n+1} = \mathbf{x}_n + \frac{\Delta t}{6}\bigl(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4\bigr)
$$

$$
\mathbf{k}_1 = f(t_n, \mathbf{x}_n), \quad
\mathbf{k}_2 = f\!\left(t_n+\tfrac{\Delta t}{2}, \mathbf{x}_n+\tfrac{\Delta t}{2}\mathbf{k}_1\right), \ldots
$$

**추력 모델 (Boost-Coast):**
- 부스트 구간 ($t < t_{burn}$): $T = 17{,}500$ N, 질량 선형 감소
- 코스트 구간 ($t \geq t_{burn}$): $T = 0$, 질량 = $m_{burnout}$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6DOF 유도탄 클래스 (Full 13-state Newton-Euler EOM)
# ─────────────────────────────────────────────────────────────────────────────

class Missile6DOF:
    """Quaternion 기반 6자유도 강체 유도탄 동역학 모델.

    상태벡터 (13 states):
        [x, y, z,  u, v, w,  q0, q1, q2, q3,  p, q_r, r]
    좌표계:
        위치  : NED (North-East-Down)
        속도  : 동체좌표계
        자세  : 쿼터니언 q = [q0, q1, q2, q3] (scalar-first)
        각속도: 동체좌표계 [p, q, r]
    참고:
        Zipfel Ch.4-6, Stevens & Lewis Ch.1
    """

    def __init__(self):
        # 질량 특성
        self.mass_0       = 85.3    # kg  이륙 질량
        self.mass_burnout = 71.5    # kg  연소 후 질량
        self.Ixx = 0.45             # kg·m²  롤
        self.Iyy = 12.8             # kg·m²  피치
        self.Izz = 12.8             # kg·m²  요 (축대칭: Iyy=Izz)

        # 추진
        self.thrust    = 17500.0    # N
        self.burn_time = 2.2        # s
        self.mass_rate = (self.mass_0 - self.mass_burnout) / self.burn_time

        # 기하
        self.S_ref  = 0.01267       # m²
        self.d_ref  = 0.127         # m
        self.length = 2.87          # m

        # 서브모델
        self.atm  = StandardAtmosphere1976()
        self.aero = MissileAerodynamics()
        self.g    = 9.80665         # m/s²

    def get_mass(self, t):
        """현재 질량 [kg]."""
        return self.mass_0 - self.mass_rate * t if t < self.burn_time else self.mass_burnout

    def get_thrust(self, t):
        """추력 [N]."""
        return self.thrust if t < self.burn_time else 0.0

    def state_derivative(self, t, state, delta_e=0.0, delta_r=0.0, delta_a=0.0):
        """13-상태 미분벡터  ẋ = f(t, x, δ).

        Args:
            t       : 현재 시각 [s]
            state   : (13,) 상태벡터
            delta_e : 승강타 편향 [rad]
            delta_r : 방향타 편향 [rad]
            delta_a : 에일러론 편향 [rad]
        Returns:
            dstate  : (13,) 미분벡터
        """
        # 상태 분해
        x_n, y_e, z_d = state[0:3]       # NED 위치
        u, v, w        = state[3:6]       # 동체 속도
        q_att          = state[6:10]      # 자세 쿼터니언
        p, q_r, r      = state[10:13]     # 동체 각속도

        # 쿼터니언 정규화
        q_att = quat_normalize(q_att)
        q0, q1, q2, q3 = q_att

        # DCM: NED → Body (C_nb),  Body → NED (C_bn = C_nb^T)
        C_nb = quat_to_dcm(q_att)
        C_bn = C_nb.T

        # 받음각, 옆미끄럼각, 속도
        alpha, beta, V = wind_angles(u, v, w)

        # 대기
        alt   = -z_d                                    # 고도 = -z_D
        _, _, rho, _ = self.atm.get_properties(max(float(alt), 0.0))
        q_bar = 0.5 * rho * V**2

        # 질량 & 추력
        m = self.get_mass(t)
        T = self.get_thrust(t)

        # 공력
        F_aero, M_aero = self.aero.get_forces_moments(
            alpha, beta, V, q_bar, p, q_r, r, delta_e, delta_r, delta_a)
        Fx_a, Fy_a, Fz_a = F_aero
        L_a, M_a, N_a    = M_aero

        # ── 중력 (쿼터니언 직접 계산, Zipfel eq. 5.19) ──────────────────────
        #   g_ned = [0, 0, g]^T  →  g_body = C_nb @ g_ned
        gx = self.g * 2.0*(q1*q3 - q0*q2)
        gy = self.g * 2.0*(q2*q3 + q0*q1)
        gz = self.g * (q0**2 - q1**2 - q2**2 + q3**2)

        # ── 병진 EOM (동체좌표계) ────────────────────────────────────────────
        du  = (Fx_a + T)/m + gx + (r*v   - q_r*w)
        dv  =  Fy_a    /m + gy + (p*w   - r*u)
        dw  =  Fz_a    /m + gz + (q_r*u - p*v)

        # ── 위치 운동학 (NED) ────────────────────────────────────────────────
        vel_ned    = C_bn @ np.array([u, v, w])
        dx, dy, dz = vel_ned

        # ── 쿼터니언 운동학 ──────────────────────────────────────────────────
        dq = quaternion_derivative(q_att, np.array([p, q_r, r]))
        dq0, dq1, dq2, dq3 = dq

        # ── 회전 EOM (Euler 방정식) ──────────────────────────────────────────
        dp  = (L_a + (self.Iyy - self.Izz)*q_r*r ) / self.Ixx
        dq_ = (M_a + (self.Izz - self.Ixx)*r*p   ) / self.Iyy
        dr  = (N_a + (self.Ixx - self.Iyy)*p*q_r ) / self.Izz

        return np.array([dx, dy, dz, du, dv, dw,
                         dq0, dq1, dq2, dq3, dp, dq_, dr])

    def rk4_step(self, t, state, dt, delta_e=0.0, delta_r=0.0, delta_a=0.0):
        """RK4 단일 스텝."""
        k1 = self.state_derivative(t,          state,              delta_e, delta_r, delta_a)
        k2 = self.state_derivative(t + 0.5*dt, state + 0.5*dt*k1, delta_e, delta_r, delta_a)
        k3 = self.state_derivative(t + 0.5*dt, state + 0.5*dt*k2, delta_e, delta_r, delta_a)
        k4 = self.state_derivative(t + dt,     state +     dt*k3, delta_e, delta_r, delta_a)
        new_state = state + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
        new_state[6:10] = quat_normalize(new_state[6:10])   # 단위제약 재정규화
        return new_state

    def simulate(self, state0, control_func=None, t_span=(0.0, 60.0), dt=0.001, subsample=10):
        """고정 스텝 RK4 궤적 시뮬레이션.

        Args:
            state0       : (13,) 초기 상태벡터
            control_func : (t, state) -> (δe, δr, δa)  또는 None (무제어)
            t_span       : (t_start, t_end)
            dt           : 적분 스텝 [s]
            subsample    : 기록 주기 (매 N 스텝)
        Returns:
            dict {'t', 'state', 'controls'}
        """
        if control_func is None:
            control_func = lambda t, s: (0.0, 0.0, 0.0)

        t_start, t_end = t_span
        t = float(t_start)
        state = np.array(state0, dtype=float)
        state[6:10] = quat_normalize(state[6:10])

        history = {'t': [], 'state': [], 'controls': []}
        step    = 0

        while t < t_end:
            if -state[2] < 0.0 and t > t_start:  # 지면 충돌
                break
            de, dr, da = control_func(t, state)
            state = self.rk4_step(t, state, dt, de, dr, da)
            t    += dt
            step += 1
            if step % subsample == 0:
                history['t'].append(t)
                history['state'].append(state.copy())
                history['controls'].append([de, dr, da])

        for k in history:
            history[k] = np.array(history[k]) if history[k] else np.empty(0)
        return history


print("Missile6DOF 클래스 정의 완료.")
missile = Missile6DOF()
print(f"  추력:     {missile.thrust:,.0f} N")
print(f"  연소시간:  {missile.burn_time} s")
print(f"  이륙질량:  {missile.mass_0} kg")
print(f"  S_ref:     {missile.S_ref} m²")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 자유 비행 시뮬레이션 (무제어: δe = δr = δa = 0)
# 초기조건: 45도 발사각, 해수면 발사
# ─────────────────────────────────────────────────────────────────────────────

# 초기 발사 조건
phi0   = 0.0                      # 롤  = 0
theta0 = np.deg2rad(45.0)         # 피치 = 45 deg (발사각)
psi0   = 0.0                      # 요  = 0 (북쪽)

V0 = 50.0                         # 초기 속도 [m/s] (발사대 속도 가정)
u0 = V0 * np.cos(theta0)          # 전방 속도
w0 = -V0 * np.sin(theta0)         # 수직 속도 (동체 z: down, 상승 = -)

q0_init = euler_to_quat(phi0, theta0, psi0)

state0_6dof = np.array([
    0.0, 0.0, 0.0,          # 위치: NED [m]
    u0,  0.0, w0,           # 속도: 동체 [m/s]
    *q0_init,               # 쿼터니언
    0.0, 0.0, 0.0,          # 각속도 [rad/s]
])

missile6 = Missile6DOF()

print("6DOF 시뮬레이션 실행 중...")
hist6 = missile6.simulate(
    state0_6dof,
    control_func=None,
    t_span=(0.0, 25.0),
    dt=0.001,
    subsample=10,
)
print(f"완료: {len(hist6['t'])} 기록 포인트, "
      f"비행시간 {hist6['t'][-1]:.2f} s")

# ── 결과 추출 ─────────────────────────────────────────────────────────────
t6   = hist6['t']
s6   = hist6['state']              # shape: (N, 13)
x6   = s6[:, 0]                    # NED North [m]
y6   = s6[:, 1]                    # NED East  [m]
z6   = s6[:, 2]                    # NED Down  [m]
alt6 = -z6                         # 고도 [m]

u6   = s6[:, 3]
v6   = s6[:, 4]
w6   = s6[:, 5]
V6   = np.sqrt(u6**2 + v6**2 + w6**2)

# 오일러각 추출
euler6 = np.array([quat_to_euler(s6[i, 6:10]) for i in range(len(t6))])
phi6   = np.rad2deg(euler6[:, 0])
theta6 = np.rad2deg(euler6[:, 1])
psi6   = np.rad2deg(euler6[:, 2])

# ── 4-패널 그림 ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('6DOF 자유 비행 시뮬레이션 결과\n(초기 발사각 45°, 무제어)', fontsize=14)

# (1) 3D 궤적
ax3d = fig.add_subplot(2, 2, 1, projection='3d')
ax3d.plot(x6/1000, y6/1000, alt6/1000, 'b-', lw=2)
ax3d.set_xlabel('북 방향 [km]')
ax3d.set_ylabel('동 방향 [km]')
ax3d.set_zlabel('고도 [km]')
ax3d.set_title('3D 비행궤적')
ax3d.scatter([0], [0], [0], color='g', s=80, zorder=5, label='발사')
ax3d.scatter([x6[-1]/1000], [y6[-1]/1000], [alt6[-1]/1000],
             color='r', s=80, zorder=5, label='충돌')
ax3d.legend()
fig.axes[0].remove()   # 기존 2D axis 제거
fig.add_axes(ax3d)

# 새 figure로 4-panel 재구성 (3D 포함)
plt.close(fig)

from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(15, 11))
fig.suptitle('6DOF 자유 비행 시뮬레이션 결과  (초기 발사각 45°, 무제어)', fontsize=13)

# 패널 1: 고도 vs 수평 거리
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(x6/1000, alt6/1000, 'b-', lw=2)
ax1.set_xlabel('수평 거리 (북) [km]')
ax1.set_ylabel('고도 [km]')
ax1.set_title('비행궤적 (수직 단면)')
ax1.scatter([0], [0], color='g', s=80, zorder=5, label='발사점')
ax1.scatter([x6[-1]/1000], [alt6[-1]/1000], color='r', s=80, zorder=5, label='착탄')
ax1.legend()
ax1.grid(True)
ax1.annotate(f"사거리: {x6[-1]/1000:.2f} km",
             xy=(x6[-1]/1000*0.5, 0.5), fontsize=10, color='navy')

# 패널 2: 속도
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(t6, V6, 'r-', lw=2, label='전속도 V')
ax2.plot(t6, u6, 'b--', lw=1.5, label='u (전방)')
ax2.plot(t6, -w6, 'g--', lw=1.5, label='-w (상향)')  # -w = 상향 분력
ax2.axvline(missile6.burn_time, color='orange', ls=':', lw=2, label=f'연소종료 t={missile6.burn_time}s')
ax2.set_xlabel('시간 [s]')
ax2.set_ylabel('속도 [m/s]')
ax2.set_title('속도 프로파일')
ax2.legend(fontsize=8)
ax2.grid(True)

# 패널 3: 자세각
ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(t6, theta6, 'b-',  lw=2, label='피치각 θ')
ax3.plot(t6, phi6,   'g--', lw=1.5, label='롤각 φ')
ax3.plot(t6, psi6,   'r:',  lw=1.5, label='요각 ψ')
ax3.axvline(missile6.burn_time, color='orange', ls=':', lw=2)
ax3.set_xlabel('시간 [s]')
ax3.set_ylabel('자세각 [deg]')
ax3.set_title('오일러각 (쿼터니언 → 오일러 변환)')
ax3.legend()
ax3.grid(True)

# 패널 4: 3D 궤적
ax4 = fig.add_subplot(2, 2, 4, projection='3d')
ax4.plot(x6/1000, y6/1000, alt6/1000, 'b-', lw=2)
ax4.scatter([0], [0], [0], color='g', s=80, zorder=5, label='발사')
ax4.scatter([x6[-1]/1000], [y6[-1]/1000], [alt6[-1]/1000],
            color='r', s=80, zorder=5, label='착탄')
ax4.set_xlabel('북 [km]')
ax4.set_ylabel('동 [km]')
ax4.set_zlabel('고도 [km]')
ax4.set_title('3D 비행궤적')
ax4.legend()

plt.tight_layout()
plt.show()

print(f"\n=== 6DOF 비행 결과 요약 ===")
print(f"비행시간:    {t6[-1]:.2f} s")
print(f"사거리:      {x6[-1]/1000:.3f} km")
print(f"최대 고도:   {alt6.max()/1000:.3f} km")
print(f"최대 속도:   {V6.max():.1f} m/s  (Mach {V6.max()/340.3:.2f})")
print(f"최종 피치각: {theta6[-1]:.1f} deg")

## 7. 3DOF vs 6DOF 비교

같은 초기조건에서 3DOF 질점 모델과 6DOF 강체 모델을 비교합니다.

**3DOF 모델 가정:**
- 유도탄 = 질점 (자세 동역학 없음)
- 속도벡터 방향 = 기체 방향 (α = 0 항상)
- 공력: $C_D(V)$만 적용 (단순화)
- 중력: NED 관성좌표계에서 직접 적용

**핵심 차이:** 6DOF에서는 발사 직후 자세 과도응답 때문에 유효 받음각이 0이 아니며,
이로 인해 초기 궤적이 3DOF와 달라집니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3DOF 질점 시뮬레이션 구현
# ─────────────────────────────────────────────────────────────────────────────

class Missile3DOF:
    """3자유도 질점 유도탄 모델.

    가정:
        - 자세 = 속도벡터 방향 (α ≡ 0)
        - 공력: 항력 CD(V)만 적용, 양력 없음 (무유도 자유 비행)
        - 추력: 6DOF와 동일한 boost-coast 모델
    상태벡터 (6 states):
        [x_N, y_E, z_D,  Vx, Vy, Vz]  (NED 관성좌표계)
    """

    def __init__(self):
        self.mass_0       = 85.3
        self.mass_burnout = 71.5
        self.thrust       = 17500.0
        self.burn_time    = 2.2
        self.mass_rate    = (self.mass_0 - self.mass_burnout) / self.burn_time
        self.S_ref        = 0.01267
        self.CD_0         = 0.35
        self.atm          = StandardAtmosphere1976()
        self.g            = 9.80665

    def get_mass(self, t):
        return self.mass_0 - self.mass_rate*t if t < self.burn_time else self.mass_burnout

    def get_thrust(self, t):
        return self.thrust if t < self.burn_time else 0.0

    def state_derivative(self, t, state):
        """6-상태 미분.  state = [xN, yE, zD, Vx, Vy, Vz]."""
        x_n, y_e, z_d = state[0:3]
        Vx, Vy, Vz    = state[3:6]
        V = np.sqrt(Vx**2 + Vy**2 + Vz**2)

        alt   = -z_d
        _, _, rho, _ = self.atm.get_properties(max(float(alt), 0.0))
        q_bar = 0.5 * rho * V**2

        m = self.get_mass(t)
        T = self.get_thrust(t)

        # 속도 방향 단위벡터 (NED)
        if V > 1e-6:
            v_hat = np.array([Vx, Vy, Vz]) / V
        else:
            v_hat = np.array([1.0, 0.0, 0.0])

        # 항력 (속도 반대 방향)
        F_drag = -q_bar * self.S_ref * self.CD_0 * v_hat

        # 추력 (속도 방향)
        F_thrust = T * v_hat

        # 중력 (NED: +z = down)
        F_grav = np.array([0.0, 0.0, m * self.g])

        acc = (F_drag + F_thrust + F_grav) / m

        return np.array([Vx, Vy, Vz, acc[0], acc[1], acc[2]])

    def simulate(self, state0, t_span=(0.0, 60.0), dt=0.001, subsample=10):
        t_start, t_end = t_span
        t     = float(t_start)
        state = np.array(state0, dtype=float)
        history = {'t': [], 'state': []}
        step    = 0

        while t < t_end:
            if -state[2] < 0.0 and t > t_start:
                break
            d = self.state_derivative(t, state)
            # RK4
            k1 = d
            k2 = self.state_derivative(t+0.5*dt, state+0.5*dt*k1)
            k3 = self.state_derivative(t+0.5*dt, state+0.5*dt*k2)
            k4 = self.state_derivative(t+dt,     state+    dt*k3)
            state = state + (dt/6.0)*(k1+2*k2+2*k3+k4)
            t    += dt
            step += 1
            if step % subsample == 0:
                history['t'].append(t)
                history['state'].append(state.copy())

        for k in history:
            history[k] = np.array(history[k]) if history[k] else np.empty(0)
        return history


# ── 3DOF 초기조건 (6DOF와 동일) ──────────────────────────────────────────
V0   = 50.0
th0  = np.deg2rad(45.0)
Vx0  = V0 * np.cos(th0)            # NED x (북)
Vz0  = -V0 * np.sin(th0)           # NED z (down, 음수 = 상승)

state0_3dof = np.array([0.0, 0.0, 0.0, Vx0, 0.0, Vz0])

missile3 = Missile3DOF()
print("3DOF 시뮬레이션 실행 중...")
hist3 = missile3.simulate(state0_3dof, t_span=(0.0, 25.0), dt=0.001, subsample=10)
print(f"완료: {len(hist3['t'])} 기록 포인트")

t3   = hist3['t']
s3   = hist3['state']
x3   = s3[:, 0]
z3   = s3[:, 2]
alt3 = -z3
V3x  = s3[:, 3]
V3z  = s3[:, 5]
V3   = np.sqrt(V3x**2 + V3z**2)

# ── 비교 플롯 ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('3DOF vs 6DOF 비행궤적 비교  (초기 발사각 45°, 무제어)', fontsize=13)

# 궤적 비교
axes[0].plot(x3/1000, alt3/1000, 'r--', lw=2.5, label='3DOF (질점 모델)')
axes[0].plot(x6/1000, alt6/1000, 'b-',  lw=2.5, label='6DOF (강체 모델)')
axes[0].set_xlabel('수평 거리 (북) [km]')
axes[0].set_ylabel('고도 [km]')
axes[0].set_title('비행궤적 비교')
axes[0].legend()
axes[0].grid(True)
dx_range = abs(x6[-1] - x3[-1])
axes[0].annotate(f"사거리 차이: {dx_range:.0f} m",
                 xy=(min(x6[-1],x3[-1])/1000*0.5, 0.1), fontsize=10, color='purple')

# 속도 비교
axes[1].plot(t3, V3, 'r--', lw=2.5, label='3DOF 전속도')
axes[1].plot(t6, V6, 'b-',  lw=2.5, label='6DOF 전속도')
axes[1].axvline(missile6.burn_time, color='orange', ls=':', lw=2,
                label=f'연소종료 ({missile6.burn_time}s)')
axes[1].set_xlabel('시간 [s]')
axes[1].set_ylabel('전속도 [m/s]')
axes[1].set_title('속도 비교')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\n=== 3DOF vs 6DOF 수치 비교 ===")
print(f"{'':20s}  {'3DOF':>12s}  {'6DOF':>12s}  {'차이':>12s}")
print(f"{'사거리 [km]':20s}  {x3[-1]/1000:>12.3f}  {x6[-1]/1000:>12.3f}  {(x6[-1]-x3[-1])/1000:>+12.3f}")
print(f"{'최대고도 [km]':20s}  {alt3.max()/1000:>12.3f}  {alt6.max()/1000:>12.3f}  {(alt6.max()-alt3.max())/1000:>+12.3f}")
print(f"{'비행시간 [s]':20s}  {t3[-1]:>12.2f}  {t6[-1]:>12.2f}  {t6[-1]-t3[-1]:>+12.2f}")
print(f"{'최대속도 [m/s]':20s}  {V3.max():>12.1f}  {V6.max():>12.1f}  {V6.max()-V3.max():>+12.1f}")
print()
print("[해석] 6DOF에서는 발사 직후 자세 과도응답으로 인해 유효 받음각이 발생합니다.")
print("       이 과도응답이 초기 궤적 차이를 만들며, 실제 유도탄 설계에서는")
print("       이 자세 과도응답을 최소화하는 오토파일럿이 필요합니다.")

## 8. 정리 (Summary)

### 13-상태 6DOF 모델 요약

| 상태 | 심볼 | 단위 | 방정식 |
|------|------|------|--------|
| NED 위치 | $x_N, y_E, z_D$ | m | $\dot{\mathbf{r}} = C_{bn}\,\mathbf{V}_b$ |
| 동체 속도 | $u, v, w$ | m/s | Newton EOM (코리올리 포함) |
| 자세 쿼터니언 | $q_0, q_1, q_2, q_3$ | – | $\dot{\mathbf{q}} = \frac{1}{2}\mathbf{q}\otimes\boldsymbol{\Omega}$ |
| 동체 각속도 | $p, q, r$ | rad/s | Euler 방정식 |

### 쿼터니언 사용의 실질적 이점

1. **특이점 없음**: 오일러각의 짐벌락($\theta = \pm 90°$) 문제 완전 회피
2. **수치 안정성**: 매 스텝 재정규화로 DCM 직교성 유지
3. **연산 효율**: DCM (9개) 대비 메모리 절약, 4원소만으로 완전 표현
4. **보간 용이**: SLERP(Spherical Linear Interpolation) 직접 적용 가능
5. **현대 표준**: INS/IMU 센서 출력이 대부분 쿼터니언 또는 각속도 형태

### 구현된 서브모델

- **좌표변환**: `euler_to_dcm`, `quat_to_dcm`, `euler_to_quat`, `quat_to_euler`
- **쿼터니언 운동학**: `quaternion_derivative`, 단위제약 재정규화
- **공력 모델**: `MissileAerodynamics` — 선형 계수, 동압 정규화
- **대기 모델**: `StandardAtmosphere1976` — 0~32 km, 3층 구조
- **6DOF 동역학**: `Missile6DOF.state_derivative` — 13-상태 Newton-Euler EOM
- **수치 적분**: RK4 고정 스텝, 매 스텝 쿼터니언 재정규화

### 다음 단계

```
다음 노트북: 05_autopilot_design.ipynb
──────────────────────────────────────────────────
6DOF + Autopilot + Guidance 통합 교전 시뮬레이션

1. 피치/요 오토파일럿 설계 (PD 또는 3루프)
   → 자세 과도응답 최소화
   → 가속도 명령 추종

2. 비례항법유도 (PN) with 6DOF 플랜트
   → 3DOF PN 대비 명중 정밀도 비교

3. HILS(Hardware-In-the-Loop Simulation) 준비
   → 실시간 C/C++ 코드 생성 고려사항
```

---

*참고: 이 노트북의 모든 수치 구현은 LIG넥스원 missile-guidance-control 저장소의*
*`src/dynamics/missile_6dof.py`, `src/dynamics/aerodynamics.py`,*
*`src/dynamics/atmosphere.py`, `src/utils/coordinate_transforms.py`와*
*동일한 수학적 구조를 따릅니다.*